In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [2]:
df = spark.sql("SELECT * FROM Instacart_basket_LH.silver_tables.orders_silver ")
display(df)

rows = df.count()
print(rows)

StatementMeta(, 15500533-addf-40e5-a077-7c8a87bcfae7, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db12455e-6c77-4e85-a289-a7b473eeff31)

3421083


In [2]:

table_names = ["aisles", "departments", "orders", "products",
               "order_products__prior", "order_products__train"]

dfs = {
    name: spark.table(f"Instacart_basket_LH.dbo.{name}")
    for name in table_names
}


StatementMeta(, 23afde75-f2ce-4326-921f-3cce7899bc6b, 4, Finished, Available, Finished, False)

In [5]:
dfs["aisles"].printSchema()

StatementMeta(, b59fb0ee-1ebb-4df7-8b68-7ecdd32bd490, 7, Finished, Available, Finished, False)

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)



In [6]:
dfs["departments"].printSchema()

StatementMeta(, b59fb0ee-1ebb-4df7-8b68-7ecdd32bd490, 8, Finished, Available, Finished, False)

root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)



In [11]:
dfs["products"].printSchema()

StatementMeta(, b59fb0ee-1ebb-4df7-8b68-7ecdd32bd490, 13, Finished, Available, Finished, False)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)



In [14]:
from pyspark.sql import functions as F

dfs["products"] = (
    dfs["products"]
    .withColumn("aisle_id", F.col("aisle_id").cast("int"))
    .withColumn("department_id", F.col("department_id").cast("int"))
)

StatementMeta(, b59fb0ee-1ebb-4df7-8b68-7ecdd32bd490, 16, Finished, Available, Finished, False)

In [15]:
dfs["products"].printSchema()

StatementMeta(, b59fb0ee-1ebb-4df7-8b68-7ecdd32bd490, 17, Finished, Available, Finished, False)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



In [19]:
dfs["products"].write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("Instacart_basket_LH.dbo.products")

StatementMeta(, b59fb0ee-1ebb-4df7-8b68-7ecdd32bd490, 24, Finished, Available, Finished, False)

In [2]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

summary = []
missing_report = []

for name, df in dfs.items():

    expressions = [F.count("*").alias("total_rows")]

    for field in df.schema.fields:
        c = F.col(field.name)

        expressions.append(
            F.count(F.when(c.isNull(), 1)).alias(f"{field.name}_null")
        )

        if isinstance(field.dataType, StringType):
            expressions.append(
                F.count(F.when(F.trim(c) == "", 1))
                .alias(f"{field.name}_blank")
            )

    stats = df.agg(*expressions).first().asDict()
    total = stats["total_rows"]

    summary.append((name, total, len(df.columns)))

    for field in df.schema.fields:
        nulls = stats[f"{field.name}_null"]
        blanks = stats.get(f"{field.name}_blank", 0)

        missing_report.append((
            name,
            field.name,
            nulls,
            blanks,
            round((nulls + blanks) * 100 / total, 2)
            if total else 0.0
        ))

display(spark.createDataFrame(
    summary, ["table", "total_rows", "total_columns"]
))

display(spark.createDataFrame(
    missing_report,
    ["table", "column", "nulls", "blanks", "missing_pct"]
))


StatementMeta(, ec34578b-a3f2-41ee-bddb-1a47f79fb147, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eab20e93-ae84-4ab9-a169-11f9376147b9)

SynapseWidget(Synapse.DataFrame, 1c42901e-daee-423c-b528-7f1fe3d20524)

In [8]:
display(
    dfs["orders"].select('eval_set')
    .distinct()
)

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c8101a73-18a8-4671-8b60-debb502954aa)

In [9]:
dfs["orders-silver"] = (
    dfs["orders"].withColumn(
        "first_order_flag",
        F.when(
            (F.col('order_number') == 1) &
            (F.col('days_since_prior_order').isNull()),
            1
        ).otherwise(0)
    )
    .withColumnRenamed('order_dow','order_day_of_week')
    
)



StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 11, Finished, Available, Finished, False)

In [10]:
display(dfs["orders-silver"])


StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f29a86f7-2bd5-424c-9d06-7abd13bfe8a7)

In [15]:
dfs["orders-silver"].write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("Instacart_basket_LH.silver_tables.orders_silver")

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 17, Finished, Available, Finished, False)

In [18]:
dfs["orders_products_silver"] = (
    dfs["order_products__prior"]
    .withColumn("source_set",F.lit("prior"))
    .unionByName(
        dfs["order_products__train"]
        .withColumn("source_set", F.lit("train"))
    )
)

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 20, Finished, Available, Finished, False)

In [24]:
display(dfs['orders_products_silver'].count())

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 26, Finished, Available, Finished, False)

33819106

In [21]:
(
    dfs["orders_products_silver"].write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema','true')
    .saveAsTable('Instacart_basket_LH.silver_tables.order_products_silver')
)

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 23, Finished, Available, Finished, False)

In [23]:
spark.table(
    'Instacart_basket_LH.silver_tables.order_products_silver'
).count()

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 25, Finished, Available, Finished, False)

33819106

In [27]:
display(
    dfs["products"]
    .filter(
        F.col('aisle_id').isNull() |
        F.col('department_id').isNull()
    )
)

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50751e9c-e2af-4f1c-9f59-ddee556f4050)

In [34]:
display(
    dfs["products"]
    .filter(
        # F.col('product_name') ==  'Scissors' 
        F.col('product_name') == '"Scotch Kids 5""'
    )
)

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89f2cd13-d34a-4972-8812-6d6a0829dd21)

In [38]:
display(
    dfs["products"]
    .select('product_id')
    .distinct()
    .count()
)

StatementMeta(, f003e964-5631-4cc7-bc5e-7972b08ec3ce, 44, Finished, Available, Finished, False)

49688

- **Products:** Profiled product names for missing values, duplicates, spelling inconsistencies, unusual characters, and repeated text.
- Created `clean_product_name` to standardize lowercase formatting, remove extra spaces, fix text encoding issues, and clean repeated descriptions while preserving the original names.
- Kept product IDs unchanged and flagged potentially similar product names for review instead of automatically merging them.
- **Aisles:** Replaced `missing` with `unknown aisle` to make the category labels clearer.
- **Departments:** Replaced `missing` with `unknown department` for better readability and consistency.
- Preserved all original IDs and table relationships, and saved the cleaned data in the Silver layer for further SQL analysis and data modeling.


In [3]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from html import unescape

df = dfs["products"]

# --------------------------------------------------
# 1. GENERAL TEXT CLEANING
# --------------------------------------------------

# Decode HTML entities such as &amp; and &#39;
decode_html = F.udf(
    lambda x: unescape(x) if x is not None else None,
    StringType()
)

clean = decode_html(F.col("product_name"))

# Remove escaped quotes
clean = F.regexp_replace(clean, r'\\"', '"')

# Lowercase, trim, and standardize whitespace
clean = F.lower(
    F.trim(
        F.regexp_replace(clean, r"[\s\p{Zs}]+", " ")
    )
)

# Remove exact repetition of a complete phrase
# Requires at least 3 words in the repeated phrase
# to avoid changing short legitimate names
clean = F.regexp_replace(
    clean,
    r"^((?:\S+\s+){2,}\S+)\s+\1$",
    "$1"
)

# Convert blank cleaned names to NULL
clean = F.when(
    F.length(clean) == 0, F.lit(None).cast("string")
).otherwise(clean)


# --------------------------------------------------
# 2. CREATE CLEAN COLUMN BESIDE ORIGINAL
# --------------------------------------------------

columns = df.columns.copy()
columns.insert(
    columns.index("product_name") + 1,
    "clean_product_name"
)

products_silver = (
    df.withColumn("clean_product_name", clean)
    .select(*columns)
)


# --------------------------------------------------
# 3. CHECK NAMES THAT NORMALIZE TO SAME VALUE
# --------------------------------------------------

same_names = (
    products_silver
    .groupBy("clean_product_name")
    .agg(
        F.collect_set("product_name").alias("original_variants")
    )
    .filter(F.size("original_variants") > 1)
)

display(same_names)


# --------------------------------------------------
# 4. DETECT POSSIBLE SPELLING VARIATIONS
# --------------------------------------------------

# Compare similar names using edit distance.
# Prefix and length filters reduce comparisons.

names = (
    products_silver
    .select("clean_product_name")
    .filter(
        F.col("clean_product_name").isNotNull() &
        (F.length("clean_product_name") >= 8)
    )
    .distinct()
    .withColumn(
        "prefix",
        F.substring("clean_product_name", 1, 4)
    )
    .withColumn(
        "name_length",
        F.length("clean_product_name")
    )
)

a = names.alias("a")
b = names.alias("b")

typo_candidates = (
    a.join(
        b,
        (F.col("a.prefix") == F.col("b.prefix")) &
        (F.col("a.clean_product_name") <
         F.col("b.clean_product_name")) &
        (F.abs(
            F.col("a.name_length") -
            F.col("b.name_length")
        ) <= 2),
        "inner"
    )
    .select(
        F.col("a.clean_product_name").alias("name_a"),
        F.col("b.clean_product_name").alias("name_b")
    )
    .withColumn(
        "edit_distance",
        F.levenshtein("name_a", "name_b")
    )
    .filter(F.col("edit_distance").between(1, 2))
)

display(
    typo_candidates
    .select("name_a", "name_b")
    .limit(30)
)


# --------------------------------------------------
# 5. APPLY REVIEWED SPELLING CORRECTIONS
# --------------------------------------------------

# Add confirmed corrections here after reviewing
# typo_candidates. No automatic fuzzy replacements.

approved_corrections = {}

for incorrect, correct in approved_corrections.items():
    products_silver = products_silver.withColumn(
        "clean_product_name",
        F.when(
            F.col("clean_product_name") == incorrect,
            F.lit(correct.lower())
        ).otherwise(F.col("clean_product_name"))
    )


# --------------------------------------------------
# 6. REVIEW ORIGINAL VS CLEANED NAMES
# --------------------------------------------------

display(
    products_silver
    .filter(
        ~F.col("product_name").eqNullSafe(
            F.col("clean_product_name")
        )
    )
    .select(
        "product_id",
        "product_name",
        "clean_product_name"
    )
    .limit(1000)
)


StatementMeta(, 23afde75-f2ce-4326-921f-3cce7899bc6b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 409cb782-ab2d-42fb-8d16-237a45350e5b)

SynapseWidget(Synapse.DataFrame, 915b3d31-5da9-4f46-bd57-df72b3cec97b)

SynapseWidget(Synapse.DataFrame, edf29cf6-fc13-43e2-aa1a-dd011832eb20)

In [6]:
display(
    products_silver
    .filter(
        F.col('aisle_id').isNull() |
        F.col('department_id').isNull()
    )
)

StatementMeta(, c5fca860-f3e3-46ce-b88b-940e59c68147, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c12df41-8870-4965-8f39-b3874866267b)

In [4]:
display(
    products_silver
    .filter(
        (F.col('aisle_id') == 100) &
        (F.col('department_id') == 21)
    )
)

StatementMeta(, ec34578b-a3f2-41ee-bddb-1a47f79fb147, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ab472ab-45b6-49bf-9754-85a264df5c28)

In [4]:
products_silver = products_silver.fillna({
    'aisle_id' : 100,
    'department_id' : 21
})

StatementMeta(, 23afde75-f2ce-4326-921f-3cce7899bc6b, 6, Finished, Available, Finished, False)

In [5]:
display(
    products_silver.filter(F.col('product_id') == 6816)
)

StatementMeta(, 23afde75-f2ce-4326-921f-3cce7899bc6b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6584c250-4753-4526-90de-3a0f47a9874b)

In [12]:

aisle_silver = (
    dfs["aisles"]
    .withColumn(
        "aisle",
        F.lower(F.trim(F.col("aisle")))
    )
    .withColumn(
        "aisle",
        F.when(
            F.col("aisle") == "missing",
            "unknown_aisle"
        ).otherwise(F.col("aisle"))
    )
)

department_silver = (
    dfs["departments"]
    .withColumn(
        "department",
        F.lower(F.trim(F.col("department")))
    )
    .withColumn(
        "department",
        F.when(
            F.col("department") == "missing",
            "unknown_department"
        ).otherwise(F.col("department"))
    )
)


StatementMeta(, ec34578b-a3f2-41ee-bddb-1a47f79fb147, 14, Finished, Available, Finished, False)

In [13]:
aisle_silver.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('Instacart_basket_LH.silver_tables.aisle_silver')
department_silver.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('Instacart_basket_LH.silver_tables.department_silver')

StatementMeta(, ec34578b-a3f2-41ee-bddb-1a47f79fb147, 15, Finished, Available, Finished, False)

In [14]:
display(department_silver)
display(aisle_silver)

StatementMeta(, ec34578b-a3f2-41ee-bddb-1a47f79fb147, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c6e1094-d9f8-4cf6-8866-269adeaf4133)

SynapseWidget(Synapse.DataFrame, 6e57c361-c286-4cae-8e80-3bf008384a5e)

In [6]:
display(products_silver)

StatementMeta(, 23afde75-f2ce-4326-921f-3cce7899bc6b, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 39fd4c44-fcad-4638-ba7b-bb9e0525f277)

In [7]:
products_silver.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('Instacart_basket_LH.silver_tables.product_silver')

StatementMeta(, 23afde75-f2ce-4326-921f-3cce7899bc6b, 9, Finished, Available, Finished, False)